In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.interpolate import griddata
from datetime import datetime

TICKER = "SPY"

def generate_interactive_surface(ticker_symbol):
    asset = yf.Ticker(ticker_symbol)
    S0 = asset.history(period="1d")['Close'].iloc[-1]
    
    # Extraction des 15 prochaines maturités pour plus de détail
    expirations = asset.options[:15]
    all_data = []
    today = datetime.now()

    print(f"📡 Récupération des données pour {ticker_symbol}...")

    for exp in expirations:
        try:
            opt = asset.option_chain(exp)
            # Filtrage des données aberrantes (IV trop basse ou pas de volume)
            calls = opt.calls[(opt.calls['impliedVolatility'] > 0.05) & (opt.calls['volume'] > 1)]
            
            t_days = (datetime.strptime(exp, '%Y-%m-%d') - today).days
            if t_days <= 0: continue
            T = t_days / 365.0
            
            for _, row in calls.iterrows():
                all_data.append([row['strike'], T, row['impliedVolatility']])
        except:
            continue

    df = pd.DataFrame(all_data, columns=['Strike', 'T', 'IV'])
    
    # Création de la grille d'interpolation
    strikes_grid = np.linspace(df['Strike'].min(), df['Strike'].max(), 60)
    times_grid = np.linspace(df['T'].min(), df['T'].max(), 60)
    S_mesh, T_mesh = np.meshgrid(strikes_grid, times_grid)
    
    # Interpolation pour lissage
    V_mesh = griddata((df['Strike'], df['T']), df['IV'], (S_mesh, T_mesh), method='cubic')

    # --- CRÉATION DU GRAPHIQUE INTERACTIF ---
    fig = go.Figure(data=[go.Surface(
        x=S_mesh, 
        y=T_mesh, 
        z=V_mesh,
        colorscale='Viridis',
        colorbar_title='IV (σ)',
        hovertemplate='Strike: %{x}<br>Years: %{y}<br>IV: %{z:.2%}<extra></extra>'
    )])

    # Ajout du point Spot actuel pour référence
    fig.add_trace(go.Scatter3d(
        x=[S0], y=[0], z=[df[df['T'] < 0.2]['IV'].mean()],
        mode='markers',
        marker=dict(size=8, color='red', symbol='diamond'),
        name='Current Spot'
    ))

    fig.update_layout(
        title=f'Volatility Surface Interactive - {ticker_symbol}',
        scene=dict(
            xaxis_title='Strike (K)',
            yaxis_title='Maturity (Years)',
            zaxis_title='Implied Volatility (σ)',
            aspectmode='manual',
            aspectratio=dict(x=1, y=1.2, z=0.5) # On étire la maturité pour mieux voir le Skew
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        template="plotly_dark" # Look "Terminal Bloomberg"
    )

    fig.show()

generate_interactive_surface(TICKER)

📡 Récupération des données pour SPY...
